In [21]:
import os,sys,requests,zipfile,shutil;
from osgeo import gdal,ogr,osr;
from osgeo_utils import gdal_calc,gdal_edit;
gdal.UseExceptions();
g_cache_gb = 96;
g_cpus     = 'ALL_CPUS';

sys.path.append(os.path.join(os.path.expanduser('~'),'notebooks'));
import common;

lddk = os.path.join(os.path.expanduser('~'),'loading_dock','grids_2026','chbay');
if not os.path.exists(lddk):
    os.mkdir(lddk);

aoi_gpkg = os.path.join(lddk,'aoi.gpkg');
outname = 'va_buff';


In [13]:
fetch_aoi = False;

if fetch_aoi is True:

    if os.path.exists(aoi_gpkg):
        os.remove(aoi_gpkg);

    srs = osr.SpatialReference();
    srs.ImportFromEPSG(5070);

    drv = ogr.GetDriverByName('GPKG');
    dst_ds = drv.CreateDataSource(aoi_gpkg);
    
    dst_lyr1 = dst_ds.CreateLayer('aoi',geom_type=ogr.wkbPolygon,srs=srs);
    nf1 = ogr.FieldDefn('aoi_id',ogr.OFTString);
    nf1.SetWidth(255);
    dst_lyr1.CreateField(nf1);    
    dst_ds = None;

    qry = '?outSR=5070&where=STUSAB%3D%27VA%27&outFields=STUSAB&f=json';
    gdal.VectorTranslate(
         srcDS            = r'https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/State_County/MapServer/0/query' + qry
        ,destNameOrDestDS = aoi_gpkg
        ,layerName        = 'aoi'
        ,format           = 'GPKG'
        ,dstSRS           = 'EPSG:5070'
        ,accessMode       = 'append'
    );

    ds = drv.Open(aoi_gpkg,1);
    ds.ExecuteSQL("UPDATE aoi SET aoi_id = 'VA'",dialect='SQLITE');
    lyr = ds.GetLayer('aoi');
    for feat in lyr:
        geom = feat.GetGeometryRef();
        if geom:
            buff_geom = geom.Buffer(8046.72);
            feat.SetGeometry(buff_geom);
            lyr.SetFeature(feat);

    ds = None;            
    

In [14]:
hucs = [
     ['67a12687d34e5b4a4f9e4e30','02050101']
    ,['679cfcf1d34e89501cd2d6c7','02050102']
    ,['679cfcf3d34e89501cd2d6c9','02050103']
    ,['679cfcf4d34e89501cd2d6cb','02050104']
    ,['679cfcf6d34e89501cd2d6cd','02050105']
    ,['679cfcf8d34e89501cd2d6cf','02050106']
    ,['679cfcfad34e89501cd2d6d1','02050107']
    
    ,['679cfcfcd34e89501cd2d6d3','02050201']
    ,['679cfcfed34e89501cd2d6d5','02050202']
    ,['679cfcffd34e89501cd2d6d7','02050203']
    ,['679cfd01d34e89501cd2d6d9','02050204']
    ,['679cfd03d34e89501cd2d6db','02050205']
    ,['679cfd05d34e89501cd2d6dd','02050206']
    
    ,['679cfd07d34e89501cd2d6df','02050301']
    ,['679cfd09d34e89501cd2d6e1','02050302']
    ,['679cfd0ad34e89501cd2d6e3','02050303']
    ,['679cfd0cd34e89501cd2d6e5','02050304']
    ,['679cfd0ed34e89501cd2d6e7','02050305']
    ,['679cfd10d34e89501cd2d6e9','02050306']
    
    ,['679cfd12d34e89501cd2d6eb','02060002']
    ,['679cfd13d34e89501cd2d6ed','02060003']
    ,['679cfd15d34e89501cd2d6ef','02060004']
    ,['679cfd17d34e89501cd2d6f1','02060005']
    ,['679cfd19d34e89501cd2d6f3','02060006']
    
    ,['679cfd1bd34e89501cd2d6f5','02070001']
    ,['679cfd1cd34e89501cd2d6f7','02070002']
    ,['679cfd1ed34e89501cd2d6f9','02070003']
    ,['679cfd20d34e89501cd2d6fb','02070004']
    ,['679cfd22d34e89501cd2d6ff','02070005']
    ,['679cfd24d34e89501cd2d701','02070006']
    ,['679cfd26d34e89501cd2d703','02070007']
    ,['679cfd28d34e89501cd2d705','02070008']
    ,['679cfd29d34e89501cd2d707','02070009']
    ,['679cfd2bd34e89501cd2d709','02070010']
    ,['679cfd2ed34e89501cd2d70c','02070011']
    
    ,['679cfd31d34e89501cd2d70f','02080102']
    ,['679cfd33d34e89501cd2d711','02080103']
    ,['679cfd35d34e89501cd2d713','02080104']
    ,['679cfd38d34e89501cd2d716','02080105']
    ,['679cfd3ad34e89501cd2d718','02080106']
    ,['679cfd3cd34e89501cd2d71a','02080107']
    ,['679cfd3ed34e89501cd2d71c','02080108']
    ,['679cfd40d34e89501cd2d71e','02080109']
    ,['679cfd42d34e89501cd2d720','02080110']
    ,['679cfd44d34e89501cd2d722','02080111']
    
    ,['679cfd45d34e89501cd2d724','02080201']
    ,['679cfd47d34e89501cd2d726','02080202']
    ,['679cfd4ad34e89501cd2d728','02080203']
    ,['679cfd4cd34e89501cd2d72a','02080204']
    ,['679cfd4ed34e89501cd2d72c','02080205']
    ,['679cfd50d34e89501cd2d72e','02080206']
    ,['679cfd52d34e89501cd2d730','02080207']
    ,['679cfd54d34e89501cd2d732','02080208']
];


In [15]:
download_chbay = False;

if download_chbay is True:
    url = r'https://prod-is-usgs-sb-prod-publish.s3.amazonaws.com/';

    for item in hucs:
        u1 = url + item[0] + '/huc_' + item[1] + '_geomorphon1m.tif';
        u2 = url + item[0] + '/huc_' + item[1] + '_geomorphon10m.tif';
        
        #print('   checking ' + u1);
        #check_link(u1);
        #print('   checking ' + u2);
        #check_link(u2);
        
        print('downloading ' + u1);
        download_link(u1,os.path.join(lddk,'huc_' + item[1] + '_geomorphon1m.tif'));
        print('downloading ' + u2);
        download_link(u2,os.path.join(lddk,'huc_' + item[1] + '_geomorphon10m.tif'));
        

In [16]:
# Step to force tiff srids to EPSG:5070 and remove dodgy colormaps

wash_tiffs = False;

if wash_tiffs is True:
    gdal.SetCacheMax(g_cache_gb * 1024 * 1024 * 1024);
    gdal.SetConfigOption('GDAL_NUM_THREADS',g_cpus);

    for item in hucs:
        print("washing " + item[1]);
        
        s1  = os.path.join(lddk,'huc_' + item[1] + '_geomorphon1m.tif');
        t1  = os.path.join(lddk,'huc_' + item[1] + '_geomorphon1m_wash.tif');

        if os.path.exists(t1):
            os.remove(t1);

        o1 = gdal.TranslateOptions(
             colorInterpretation = ['Gray']
            ,format              = 'GTiff'
            ,outputSRS           = 'EPSG:5070'
            ,creationOptions     = ['COMPRESS=DEFLATE','TILED=YES','TFW=YES']
        );            
        ds = gdal.Translate(
             srcDS       = s1
            ,destName    = t1
            ,options     = o1
        );

        s10 = os.path.join(lddk,'huc_' + item[1] + '_geomorphon10m.tif'); 
        t10 = os.path.join(lddk,'huc_' + item[1] + '_geomorphon10m_wash.tif'); 

        if os.path.exists(t10):
            os.remove(t10);

        o10 = gdal.TranslateOptions(
             colorInterpretation = ['Gray']
            ,format              = 'GTiff'
            ,outputSRS           = 'EPSG:5070'
            ,creationOptions     = ['COMPRESS=DEFLATE','TILED=YES','TFW=YES']
        ); 
        ds = gdal.Translate(
             srcDS       = s10
            ,destName    = t10
            ,options     = o10
        );
        

In [17]:
# This step can also be used to verify all washed tifs exist
create_info = False;

if create_info is True:

    for item in hucs:

        t1  = os.path.join(lddk,'huc_' + item[1] + '_geomorphon1m_wash.tif');
        i1  = os.path.join(lddk,'huc_' + item[1] + '_geomorphon1m_wash.gdalinfo');

        t10 = os.path.join(lddk,'huc_' + item[1] + '_geomorphon10m_wash.tif');
        i10 = os.path.join(lddk,'huc_' + item[1] + '_geomorphon10m_wash.gdalinfo');

        if os.path.exists(i1):
            os.remove(i1);

        with gdal.OpenEx(t1,allowed_drivers=['GTiff']) as ds:

            with open(i1,"w") as f:
                f.write(gdal.Info(ds));

        if os.path.exists(i10):
            os.remove(i10);

        with gdal.OpenEx(t10,allowed_drivers=['GTiff']) as ds:

            with open(i10,"w") as f:
                f.write(gdal.Info(ds));  
                

In [18]:
# Boolean flag to generate raster footprints 

build_footprints = False;

if build_footprints is True:
    gdal.SetCacheMax(g_cache_gb * 1024 * 1024 * 1024);
    gdal.SetConfigOption('GDAL_NUM_THREADS','ALL_CPUS');
    
    if os.path.exists('/tmp/tmp1.tif'):
        os.remove('/tmp/tmp1.tif');
    if os.path.exists('/tmp/tmp10.tif'):
        os.remove('/tmp/tmp10.tif');
        
    ftprtgpkg = os.path.join(lddk,'chbay_footprints.gpkg');
    if os.path.exists(ftprtgpkg):
        os.remove(ftprtgpkg);

    srs = osr.SpatialReference();
    srs.ImportFromEPSG(5070);
                
    for item in hucs:
        t1  = os.path.join(lddk,'huc_' + item[1] + '_geomorphon1m_wash.tif');
        t10 = os.path.join(lddk,'huc_' + item[1] + '_geomorphon10m_wash.tif');
            
        print('creating temp byte mask for 1m ' + item[1]);
        gdal_calc.Calc(
             calc        = "where(A<255,1,0)"
            ,A           = t1
            ,outfile     = '/tmp/tmp1.tif'
            ,NoDataValue = 0
            ,format      = 'GTiff'
            ,type        = 'Byte'
            ,overwrite   = True
            ,creation_options = ['TFW=YES']
        );

        src_ds1 = gdal.OpenEx('/tmp/tmp1.tif',allowed_drivers=['GTiff']);
        src_bnd1 = src_ds1.GetRasterBand(1);
        src_msk1 = src_bnd1.GetMaskBand();

        print('creating temp byte mask for 10m ' + item[1]);
        gdal_calc.Calc(
             calc        = "where(A<255,1,0)"
            ,A           = t10
            ,outfile     = '/tmp/tmp10.tif'
            ,NoDataValue = 0
            ,format      = 'GTiff'
            ,type        = 'Byte'
            ,overwrite   = True
            ,creation_options = ['TFW=YES']
        );

        src_ds10 = gdal.OpenEx('/tmp/tmp10.tif',allowed_drivers=['GTiff']);
        src_bnd10 = src_ds10.GetRasterBand(1);
        src_msk10 = src_bnd10.GetMaskBand();

        try:
            gdal.PushErrorHandler('CPLQuietErrorHandler');
            dst_ds = ogr.Open(ftprtgpkg,update=1);
            gdal.PopErrorHandler();
        except:
            dst_ds = None;

        if dst_ds is None:
            print("creating new geopackage");
            drv = ogr.GetDriverByName('GPKG');
            dst_ds = drv.CreateDataSource(ftprtgpkg);

        try:
            dst_lyr1 = dst_ds.GetLayerByName('footprints_1m');
        except:
            dst_lyr1 = None;
        
        if dst_lyr1 is None:
            print("adding new footprints_1m layer");
            dst_lyr1 = dst_ds.CreateLayer('footprints_1m',geom_type=ogr.wkbPolygon,srs=srs);
            print("adding new gid field");
            nf1 = ogr.FieldDefn('gid',ogr.OFTInteger);
            dst_lyr1.CreateField(nf1);
            print("adding new huc8 field");
            nf2 = ogr.FieldDefn('huc8',ogr.OFTString)
            nf2.SetWidth(8);
            dst_lyr1.CreateField(nf2);
            dst_field1 = 0;

        try:
            dst_lyr10 = dst_ds.GetLayerByName('footprints_10m');
        except:
            dst_lyr10 = None;
        
        if dst_lyr10 is None:
            print("adding new footprints_10m layer");
            dst_lyr10 = dst_ds.CreateLayer('footprints_10m',geom_type=ogr.wkbPolygon,srs=srs);
            print("adding new gid field");
            nf1 = ogr.FieldDefn('gid',ogr.OFTInteger);
            dst_lyr10.CreateField(nf1);
            print("adding new huc8 field");
            nf2 = ogr.FieldDefn('huc8',ogr.OFTString)
            nf2.SetWidth(8);
            dst_lyr10.CreateField(nf2);
            dst_field10 = 0;    

        print('polygonizing 1m result');  
        gdal.Polygonize(src_bnd1,src_msk1,dst_lyr1,dst_field1,[],callback=None);

        sql = "UPDATE main.footprints_1m  SET gid = " + str(item[1]) + ",huc8 = '" + str(item[1]) + "' WHERE gid = 1";
        rez = dst_ds.ExecuteSQL(sql);

        print('polygonizing 10m result');  
        gdal.Polygonize(src_bnd10,src_msk10,dst_lyr10,dst_field10,[],callback=None);

        sql = "UPDATE main.footprints_10m SET gid = " + str(item[1]) + ",huc8 = '" + str(item[1]) + "' WHERE gid = 1";
        rez = dst_ds.ExecuteSQL(sql);
            
    dst_lyr = None;
    dst_ds = None;

In [19]:
# Boolean flag to build gridset VRTs 

build_vrt = False;

if build_vrt is True:

    vrt1  = os.path.join(lddk,'geomorphon1m_wash.vrt');
    vrt10 = os.path.join(lddk,'geomorphon10m_wash.vrt');

    lst1  = os.path.join(lddk,'geomorphon1m_wash_list.txt');
    lst10 = os.path.join(lddk,'geomorphon10m_wash_list.txt');

    if os.path.exists(vrt1):
        os.remove(vrt1);

    if os.path.exists(vrt10):
        os.remove(vrt10);

    if os.path.exists(lst1):
        os.remove(lst1);

    if os.path.exists(lst10):
        os.remove(lst10);

    ary1 = [];
    with open(lst1,"w") as f:
        for item in hucs:
            t1  = os.path.join(lddk,'huc_' + item[1] + '_geomorphon1m_wash.tif');
            f.write(t1 + '\n');
            ary1.append(t1);

    ary10 = [];
    with open(lst10,"w") as f:
        for item in hucs:
            t10  = os.path.join(lddk,'huc_' + item[1] + '_geomorphon10m_wash.tif');
            f.write(t10 + '\n');
            ary10.append(t10);

    o1 = gdal.BuildVRTOptions(
         xRes = 1
        ,yRes = 1
        ,srcNodata = 255
        ,VRTNodata = 255
    );
    v1 = gdal.BuildVRT(
         destName        = vrt1
        ,srcDSOrSrcDSTab = ary1
        ,options         = o1
    );
    v1 = None;

    o10 = gdal.BuildVRTOptions(
         xRes = 10
        ,yRes = 10
        ,srcNodata = 255
        ,VRTNodata = 255
    );
    v10 = gdal.BuildVRT(
         destName        = vrt10
        ,srcDSOrSrcDSTab = ary10
        ,options         = o10
    );
    v10 = None;


In [23]:
# Boolean flag to mosaic and clip to area of interest

clip_to_aoi = True;

if clip_to_aoi is True:
    gdal.SetCacheMax(g_cache_gb * 1024 * 1024 * 1024);
    gdal.SetConfigOption('GDAL_NUM_THREADS','ALL_CPUS');
    
    vrt1  = os.path.join(lddk,'geomorphon1m_wash.vrt');
    vrt10 = os.path.join(lddk,'geomorphon10m_wash.vrt');

    clp1  = os.path.join(lddk,outname + '_geomorphon1m_wash.tif');
    clp10 = os.path.join(lddk,outname + '_geomorphon10m_wash.tif');

    if os.path.exists(clp1):
        os.remove(clp1);

    if os.path.exists(clp10):
        os.remove(clp10);
        
    o1 = gdal.WarpOptions(
         format           = 'GTiff'
        ,cutlineDSName    = aoi_gpkg
        ,cutlineLayer     = 'aoi'
        ,cropToCutline    = True
        ,dstNodata        = 255
        ,creationOptions  = ['TFW=YES','COMPRESS=DEFLATE','TILED=YES','BIGTIFF=YES']
    );
    gdal.Warp(
         srcDSOrSrcDSTab  = vrt1
        ,destNameOrDestDS = clp1
        ,options          = o1
    );
        
    o10 = gdal.WarpOptions(
         format           = 'GTiff'
        ,cutlineDSName    = aoi_gpkg
        ,cutlineLayer     = 'aoi'
        ,cropToCutline    = True
        ,dstNodata        = 255
        ,creationOptions  = ['TFW=YES','COMPRESS=DEFLATE','TILED=YES','BIGTIFF=YES']
    );
    gdal.Warp(
         srcDSOrSrcDSTab  = vrt10 
        ,destNameOrDestDS = clp10
        ,options          = o10
    );
        